# C2 exp01: are we handicapping the decoder?

The experiment of `tasks/C2-decoder-baseline/task.md`, Part 3. Three settings on **Qwen2.5-0.5B**, with the encoder's data, k = 0, seeds, 3 epochs and epoch selection:

| arm | trains | lr | answers |
|---|---|---|---|
| `lora_r16` | adapters + head (2.17 M, 0.44 %) | 2e-4 | the decoder baseline |
| `lora_all_linear` | adapters + head (8.80 M, 1.78 %) | 2e-4 | whether a gap, if any, is a capacity limit |
| `full` | all 494 M parameters | 1e-5 | how much LoRA costs on this task |

Seeds 42, 43, 44: **9 runs**. They are queued seed by seed, so stopping early still leaves a complete comparison on the seeds that finished. To run fewer, shorten `SEEDS` in section 5.
Note: `tasks/C2-decoder-baseline/experiments/exp01_lora-vs-full/NOTE.md`.

Temporary trial (`scripts/trial_run.py --mode flat`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings:
- **Accelerator:** GPU **T4 x2**: two independent runs at a time, one per GPU
- **Internet:** on (to clone the repo and download the model)
- **Input:** the private `smi-v4-meetings` dataset attached

The repo branch `C2/decoder-baseline` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [1]:
!nvidia-smi

Sun Nov  1 21:34:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Code and packages
Do **not** install `requirements.txt` here: it would replace Kaggle's own torch build. `peft` is the one package the encoder notebooks did not need.

In [2]:
!rm -rf /kaggle/working/repo
!git clone -b C2/decoder-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 sentencepiece==0.2.2 peft==0.21.2
# Kaggle ships torchao 0.10 , and peft raises on anything below 0.16 as soon as it finds it ; nothing here uses torchao
!pip uninstall -q -y torchao

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1022, done.
remote: Counting objects: 100% (382/382), done.
remote: Compressing objects: 100% (231/231), done.
remote: Total 1022 (delta 201), reused 259 (delta 101), pack-reused 640 (from 1)
Receiving objects: 100% (1022/1022), 865.48 KiB | 16.64 MiB/s, done.
Resolving deltas: 100% (499/499), done.
/kaggle/working/repo
8a0d439 (HEAD -> C2/decoder-baseline, origin/C2/decoder-baseline) Merge C2/decoder-baseline: task description
8a57e38 C2: decoder options in the shared model code, tests, Kaggle notebooks
d2f9369 (origin/main, origin/HEAD) Add CODEOWNERS file for repository ownership
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`. Only `train/` and `val/` are read.

In [3]:
import json
import glob, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

# the same per-meeting files the encoder used : scripts/make_meeting_files.py over release v4
print("data version:", json.load(open(hits[0]))["data_version"])
for split, meetings in (("train", 51), ("val", 12)):
    found = len(glob.glob(f"{DATA_ROOT}/{split}/*.jsonl"))
    print(split, "meetings:", found)
    assert found == meetings, f"expected {meetings} meeting files in {split}/ : is this the right dataset?"

source.json found at: ['/kaggle/input/datasets/mohammedadelguest/smiiii-v4-meetings/v4_meetings/source.json']
DATA_ROOT = /kaggle/input/datasets/mohammedadelguest/smiiii-v4-meetings/v4_meetings -> ['source.json', 'test', 'train', 'val']
data version: v4
train meetings: 51
val meetings: 12


## 4. The model loads as the design says
Also downloads the weights once (988 MB). These are the mistakes that fail silently, so each one stops the notebook here rather than after hours of training: the wrong pad id, a suffix that doesn't end in the token the head reads, an fp16 trainable, a frozen head.

In [4]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")
sys.path.insert(0, "/kaggle/working/repo/scripts")
import torch
from smi.inputs.collator import SegmentCollator
from smi.models.model import count_parameters, load_model_and_tokenizer
from trial_run import DECODER_SETTINGS

MODEL = "Qwen/Qwen2.5-0.5B"
model, tokenizer = load_model_and_tokenizer(MODEL, dtype=torch.float16, lora=DECODER_SETTINGS["lora"])

print("pad id:", tokenizer.pad_token_id, model.config.pad_token_id)
assert tokenizer.pad_token_id == model.config.pad_token_id == 151643

trainable, total = count_parameters(model)
print(f"trainable: {trainable:,} of {total:,} ({100 * trainable / total:.3f} %)")
assert (trainable, total) == (2168960, 494039040)

print("trainable dtypes:", {p.dtype for p in model.parameters() if p.requires_grad})
print("frozen dtypes:", {p.dtype for p in model.parameters() if not p.requires_grad})

collator = SegmentCollator(tokenizer, 512, "\n", markers=("[T] ", " [/T]"), suffix=DECODER_SETTINGS["suffix"])
item = {"seg_id": "check", "label": 0, "left": [], "target_text": "sorry .", "right": []}
tokens = tokenizer.convert_ids_to_tokens(collator([item])["input_ids"][0])
print("tokens:", tokens)
assert tokens[-1] == "]:"
del model

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


pad id: 151643 151643
trainable: 2,168,960 of 494,039,040 (0.439 %)
trainable dtypes: {torch.float32}
frozen dtypes: {torch.float16}
tokens: ['[T', ']', 'Ġsorry', 'Ġ.', 'Ġ[/', 'T', ']Ċ', 'Label', 'Ġof', 'Ġ[', 'T', ']:']


## 5. Settings and the memory check
One training step (fp16 + fused AdamW, as in `train()`) on the longest training batch, for each arm, in a separate process so the GPU is free again afterwards. Two steps, because AdamW creates its `m` and `v` on the first one. It also confirms that the logits stay finite with the base in fp16.

LoRA should fit easily. **Full fine-tuning is the one to watch.** If an arm's peak reserved memory is above ~13.5 GB, don't run section 6: set that arm's `"grad_checkpointing": True` below and rerun this section. Checkpointing keeps the batches and the gradients the same, only slower (C1 exp05 lost a run by skipping this).

In [5]:
EPOCHS = 3
SEEDS = [42, 43, 44]
ARMS = {
    "lora_r16": {"lr": 2e-4, "flags": ["--family", "decoder"], "grad_checkpointing": False},
    "lora_all_linear": {"lr": 2e-4, "flags": ["--family", "decoder", "--lora-targets", "all-linear"], "grad_checkpointing": False},
    "full": {"lr": 1e-5, "flags": ["--family", "decoder", "--full-finetuning"], "grad_checkpointing": True},}
ENCODER_REFERENCE = {42: 0.5250, 43: 0.5267, 44: 0.5157}  # C1 exp03 : DeBERTa , lr 1e-5 , 3 epochs , k = 0


def trial_arguments(arm):
    """the flags of one arm ; the memory check and the run take the same ones , so the check measures that run."""
    setting = ARMS[arm]
    arguments = ["--mode", "flat", "--lr", str(setting["lr"]), "--epochs", str(EPOCHS), "--data-root", DATA_ROOT,
                 "--model", MODEL, "--out-dir", "/kaggle/working/runs"] + setting["flags"]
    if setting["grad_checkpointing"]:
        arguments.append("--grad-checkpointing")
    return arguments

In [6]:
%%writefile /kaggle/working/memory_check.py
"""one real training step on the longest training batch ; prints the peak GPU memory. takes trial_run.py's flags."""
import sys

import torch

sys.path.insert(0, "src")
sys.path.insert(0, "scripts")
from smi.inputs.loader import build_dataloader
from smi.models.model import build_optimizer
from trial_run import SETTINGS, apply_arguments, load_model, parse_arguments

args = parse_arguments()
apply_arguments(args)
device = torch.device("cuda")
model, tokenizer = load_model(args.model)
model.to(device)
loader = build_dataloader(
    f"{args.data_root}/train", tokenizer, past_k=SETTINGS["past_k"], future_k=SETTINGS["future_k"],
    batch_size=SETTINGS["batch_size"], chunk_batches=SETTINGS["chunk_batches"], shuffle=True,
    max_length=SETTINGS["max_length"], separator=SETTINGS["separator"], markers=tuple(SETTINGS["markers"]),
    suffix=SETTINGS["suffix"], seed=42,
)

# the batch with the most tokens (batch size × padded length)
longest = None
for batch in loader:
    if longest is None or batch["input_ids"].numel() > longest["input_ids"].numel():
        longest = batch
print("longest training batch (segments, tokens):", tuple(longest["input_ids"].shape))

optimizer = build_optimizer(model, SETTINGS["lr"], SETTINGS["weight_decay"], fused=True)
scaler = torch.amp.GradScaler("cuda")
loss_function = torch.nn.CrossEntropyLoss()
model.train()
torch.cuda.reset_peak_memory_stats()
for step in range(2):
    input_ids = longest["input_ids"].to(device)
    attention_mask = longest["attention_mask"].to(device)
    labels = longest["labels"].to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    assert torch.isfinite(logits).all(), "non-finite logits : fp16 overflow in the frozen base"
    loss = loss_function(logits.float(), labels)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), SETTINGS["max_grad_norm"])
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
torch.cuda.synchronize()

total = torch.cuda.get_device_properties(0).total_memory / 1e9
reserved = torch.cuda.max_memory_reserved() / 1e9
print(f"family {SETTINGS['family']} | lora {SETTINGS['lora'] is not None} | grad checkpointing {SETTINGS['grad_checkpointing']} | "
      f"peak allocated {torch.cuda.max_memory_allocated() / 1e9:.1f} GB , peak reserved {reserved:.1f} GB , GPU total {total:.1f} GB")
if reserved > 13.5:
    print("TOO CLOSE : set this arm's grad_checkpointing to True and run the settings cell and this check again")
else:
    print("OK : batch 32 fits")

Writing /kaggle/working/memory_check.py


In [7]:
import subprocess

for arm in ARMS:
    print("=====", arm, flush=True)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false")
    result = subprocess.run(["python", "/kaggle/working/memory_check.py"] + trial_arguments(arm), cwd="/kaggle/working/repo",
                            env=env, capture_output=True, text=True)
    print("\n".join(line for line in result.stdout.splitlines() if "fits" in line or "peak" in line or "longest" in line or "TOO CLOSE" in line))
    if result.returncode != 0:
        print(result.stderr[-3000:])  # the end of the error , e.g. out of memory

===== lora_r16
longest training batch (segments, tokens): (32, 151)
family decoder | lora True | grad checkpointing False | peak allocated 7.5 GB , peak reserved 8.0 GB , GPU total 15.6 GB
OK : batch 32 fits
===== lora_all_linear
longest training batch (segments, tokens): (32, 151)
family decoder | lora True | grad checkpointing False | peak allocated 10.2 GB , peak reserved 10.5 GB , GPU total 15.6 GB
OK : batch 32 fits
===== full
longest training batch (segments, tokens): (32, 151)
family decoder | lora False | grad checkpointing True | peak allocated 8.0 GB , peak reserved 9.3 GB , GPU total 15.6 GB
OK : batch 32 fits


## 6. Run: a queue over both T4s
Each GPU takes the next run as soon as its current one ends. The queue goes seed by seed, in the order of `ARMS`. Each process sees only one GPU (`CUDA_VISIBLE_DEVICES`), so there is no communication between them. Progress: the newest line of each running log every 2 minutes.

**Safe to rerun after a restart:**
- a run whose folder has a `metrics.json` is finished and is skipped;
- a folder without one is an interrupted run: it is deleted and the run starts again.

In [8]:
import shutil, time

RUNS_DIR = "/kaggle/working/runs"
LOGS_DIR = "/kaggle/working/logs"
os.makedirs(LOGS_DIR, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run_name(arm):
    return f"exp01_{arm}"


def start(gpu, arm, seed):
    """one training process that sees only one GPU (CUDA_VISIBLE_DEVICES) ; its output goes to its own log file."""
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    log_path = f"{LOGS_DIR}/{run_name(arm)}_seed{seed}_gpu{gpu}.log"
    cmd = ["python", "scripts/trial_run.py", "--seeds", str(seed), "--name", run_name(arm)] + trial_arguments(arm)
    process = subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env,
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    return process, log_path


# what is left to run
todo = []
for seed in SEEDS:
    for arm in ARMS:
        run_dir = f"{RUNS_DIR}/{run_name(arm)}_seed{seed}"
        if os.path.exists(f"{run_dir}/metrics.json"):
            print("finished already, skipped:", os.path.basename(run_dir))
            continue
        if os.path.exists(run_dir):
            print("interrupted run, deleted and started again:", os.path.basename(run_dir))
            shutil.rmtree(run_dir)
        todo.append((arm, seed))

# running[gpu] = (process , log path) , or None when that GPU is free
running = {0: None, 1: None}
while len(todo) > 0 or any(job is not None for job in running.values()):
    for gpu in running:
        job = running[gpu]
        if job is not None and job[0].poll() is not None:
            process, log_path = job
            print(f"{os.path.basename(log_path)} finished with exit code {process.returncode}", flush=True)
            if process.returncode != 0:
                print(open(log_path).read()[-3000:], flush=True)  # the end of the log , to see the error
            running[gpu] = None
        if running[gpu] is None and len(todo) > 0:
            arm, seed = todo.pop(0)
            running[gpu] = start(gpu, arm, seed)
            print(f"GPU {gpu}: started {run_name(arm)}_seed{seed}", flush=True)
    time.sleep(120)
    for gpu, job in running.items():
        if job is not None:
            lines = open(job[1]).read().strip().splitlines()
            print(f"{os.path.basename(job[1])}: {lines[-1] if lines else '(no output yet)'}", flush=True)
print("all runs finished")

GPU 0: started exp01_lora_r16_seed42
GPU 1: started exp01_lora_all_linear_seed42
exp01_lora_r16_seed42_gpu0.log:   step 600/7398 (epoch 1) | loss 0.5055 | lr 1.62e-04 | grad_norm 22.55 | loss scale 256 | 1.6 min
exp01_lora_all_linear_seed42_gpu1.log:   step 450/7398 (epoch 1) | loss 0.4344 | lr 1.22e-04 | grad_norm 8.14 | loss scale 256 | 1.6 min
exp01_lora_r16_seed42_gpu0.log:   step 1350/7398 (epoch 1) | loss 0.4246 | lr 1.82e-04 | grad_norm 10.72 | loss scale 256 | 3.6 min
exp01_lora_all_linear_seed42_gpu1.log:   step 1000/7398 (epoch 1) | loss 0.4383 | lr 1.92e-04 | grad_norm 11.57 | loss scale 256 | 3.6 min
exp01_lora_r16_seed42_gpu0.log:   step 2100/7398 (epoch 1) | loss 0.3733 | lr 1.59e-04 | grad_norm 6.59 | loss scale 512 | 5.7 min
exp01_lora_all_linear_seed42_gpu1.log:   step 1550/7398 (epoch 1) | loss 0.4261 | lr 1.76e-04 | grad_norm 4.84 | loss scale 256 | 5.5 min
exp01_lora_r16_seed42_gpu0.log:   step 2600/7398 (epoch 2) | loss 0.3909 | lr 1.44e-04 | grad_norm 4.33 | loss 

## 7. Results
Per arm: val macro-F1 (6) per seed, the mean ± std, and the gap to full fine-tuning **seed by seed**: the same seed gives the same batches in every arm, so these pairs are the fair comparison. Then what every decoder result must report (task.md): trainable parameters, r, α, target modules, LoRA or full. Then per run: per-class F1, val curve, time per epoch, truncation and the health of the loss scale.

A single `nan` grad_norm early on is the loss scaler skipping one overflowed step, which is normal. Many of them, or a loss scale that keeps falling, is not.

Encoder reference (C1 exp03, DeBERTa, lr 1e-5, 3 epochs, k = 0) = 0.5225 ± 0.0059.

In [9]:
import math, statistics

scores = {}
for arm in ARMS:
    scores[arm] = {}
    for seed in SEEDS:
        path = f"{RUNS_DIR}/{run_name(arm)}_seed{seed}/metrics.json"
        if os.path.exists(path):
            scores[arm][seed] = json.load(open(path))["primary"]["macro_f1_6"]

for arm in ARMS:
    line = " | ".join(f"seed {seed} {score:.4f}" for seed, score in scores[arm].items())
    print(f"{arm}: {line or 'no finished run'}")
    if len(scores[arm]) > 1:
        print(f"  mean {statistics.mean(scores[arm].values()):.4f} ± {statistics.stdev(scores[arm].values()):.4f} over {len(scores[arm])} seeds")

print("\ngap to full fine-tuning, seed by seed (arm - full):")
for arm in ("lora_r16", "lora_all_linear"):
    shared = [seed for seed in scores[arm] if seed in scores["full"]]
    gaps = [scores[arm][seed] - scores["full"][seed] for seed in shared]
    if gaps:
        print(f"  {arm}: {' , '.join(f'{gap:+.4f}' for gap in gaps)} | mean {statistics.mean(gaps):+.4f}")

print("\ndecoder vs encoder (C1 exp03), seed by seed (arm - encoder):")
for arm in ARMS:
    gaps = [score - ENCODER_REFERENCE[seed] for seed, score in scores[arm].items()]
    if gaps:
        print(f"  {arm}: {' , '.join(f'{gap:+.4f}' for gap in gaps)} | mean {statistics.mean(gaps):+.4f}")

print()
for arm in ARMS:
    for seed in SEEDS:
        run_dir = f"{RUNS_DIR}/{run_name(arm)}_seed{seed}"
        if not os.path.exists(f"{run_dir}/metrics.json"):
            continue
        m = json.load(open(f"{run_dir}/metrics.json"))
        settings = json.load(open(f"{run_dir}/settings.json"))
        parameters = json.load(open(f"{run_dir}/flat/parameters.json"))
        print(f"=== {os.path.basename(run_dir)} | selected epoch {m['selected_epoch']} "
              f"| macro_f1_6 {m['primary']['macro_f1_6']:.4f} | macro_f1_7 {m['macro_f1_7']:.4f}")
        print(f"trainable {parameters['trainable']:,} of {parameters['total']:,} ({100 * parameters['fraction']:.3f} %) "
              f"| lora {settings['lora']} | base {settings['base_dtype']} | lr {settings['lr']} "
              f"| grad checkpointing {settings['grad_checkpointing']}")
        print(f"commit {settings['commit'][:7]} | {settings['versions']}")
        print("per-class F1:", {c: round(v["f1"], 3) for c, v in m["per_class"].items()})
        curve = []
        minutes = []
        loss_scales = []
        skipped = 0
        for line in open(f"{run_dir}/flat/history.jsonl"):
            h = json.loads(line)
            if h["type"] == "epoch":
                curve.append(f"{h['val_macro_f1']:.3f}")
                minutes.append(f"{h['seconds'] / 60:.1f}")
            else:
                loss_scales.append(h["loss_scale"])
                skipped = skipped + (0 if math.isfinite(h["grad_norm"]) else 1)
        print("val macro F1 per epoch:", " , ".join(curve), "| minutes per epoch:", " , ".join(minutes))
        print(f"loss scale min {min(loss_scales):.0f} , last {loss_scales[-1]:.0f} | non-finite grad_norm in {skipped} of {len(loss_scales)} logged steps")
        for log_path in glob.glob(f"{LOGS_DIR}/{run_name(arm)}_seed{seed}_gpu*.log"):
            for line in open(log_path):
                if "truncated train inputs" in line:
                    print(line.strip())
        print()

lora_r16: seed 42 0.4924 | seed 43 0.4756 | seed 44 0.4741
  mean 0.4807 ± 0.0102 over 3 seeds
lora_all_linear: seed 42 0.4863 | seed 43 0.4644 | seed 44 0.4873
  mean 0.4793 ± 0.0130 over 3 seeds
full: seed 42 0.4617 | seed 43 0.4857 | seed 44 0.4709
  mean 0.4728 ± 0.0122 over 3 seeds

gap to full fine-tuning, seed by seed (arm - full):
  lora_r16: +0.0308 , -0.0102 , +0.0031 | mean +0.0079
  lora_all_linear: +0.0246 , -0.0214 , +0.0164 | mean +0.0065

decoder vs encoder (C1 exp03), seed by seed (arm - encoder):
  lora_r16: -0.0326 , -0.0511 , -0.0416 | mean -0.0418
  lora_all_linear: -0.0387 , -0.0623 , -0.0284 | mean -0.0432
  full: -0.0633 , -0.0410 , -0.0448 | mean -0.0497

=== exp01_lora_r16_seed42 | selected epoch 3 | macro_f1_6 0.4924 | macro_f1_7 0.5551
trainable 2,168,960 of 494,039,040 (0.439 %) | lora {'r': 16, 'alpha': 32, 'dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj']} | base fp16 | lr 0.0002 | grad checkpointing False
commit 8a0d439 | {'torc

## 8. Download the run folders
One zip with every exp01 run folder (settings, parameters, history, predictions, metrics) and the logs, in the notebook's output files.

In [10]:
!cd /kaggle/working && zip -qr c2_exp01_runs.zip runs/exp01_* logs && ls -lh c2_exp01_runs.zip

-rw-r--r-- 1 root root 9.8M Nov  2 00:06 c2_exp01_runs.zip
